### Improving reasoning-with inference time scaling

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)



Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
from evaluating_reasoning_models.generate_text_streams import generate_text_stream_with_kv_cache
from evaluating_reasoning_models.render_prompt import render_prompt


In [3]:
raw_prompt = ("Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?")

prompt = render_prompt(raw_prompt)

print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [4]:
from evaluating_reasoning_models.complete_boxed import (
    has_complete_boxed_answer,
)


def generate_text_stream_concat_flex(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens,
    verbose=False,
    generate_func=None,
    stop_after_boxed=True,
    stop_texts=("\nQuestion:", "\nAnswer:"),
    **generate_kwargs,
):
    if generate_func is None:
        generate_func = generate_text_stream_with_kv_cache

    generated_ids = []
    generated_text = ""

    for token in generate_func(
        prompt=prompt,
        model=model,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        **generate_kwargs,
    ):

        token_id = token.squeeze(0).item()
        generated_ids.append(token_id)

        token_text = tokenizer.decode([token_id])
        generated_text += token_text

        if verbose:
            print(
                token_text,
                end="",
                flush=True,
            )

        if stop_after_boxed and has_complete_boxed_answer(
            generated_text
        ):
            break

        if stop_texts and any(stop_text in generated_text for stop_text in stop_texts):
            break

    return generated_text

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
response = generate_text_stream_concat_flex(model, 
    tokenizer, 
    prompt, 
    device, 
    max_new_tokens=200, 
    verbose=True, 
    generate_func=generate_text_stream_with_kv_cache)

 \boxed{12}



### Now, using Chain-of-thought prompt ---for better response

In [6]:
prompt_cot = prompt + " \n\nExplain step step by step."

response = generate_text_stream_concat_flex(model, 
    tokenizer, 
    prompt_cot, 
    device, 
    max_new_tokens=200, 
    verbose=True)

:

Step 1:

 Let $x$ be the value we are looking for.

Step 2: The problem states that half the value of $3x - 9$ is $x + 37$.

Step 3: So, we can write the equation as $ \frac{1}{2}(3x - 9) = x + 37 $.

Step 4: Multiply both sides of the equation by 2 to eliminate the fraction.

Step 5: $ \frac{1}{2}(3x - 9) \times 2 = (x + 37) \times 2 $.

Step 6: Simplify both sides of the equation.

Step 7: $ 3x - 9 = 2x + 74 $.

Step 8: Subtract $2x$ from both sides to isolate the variable term.

Step 9: $ 3x - 2x - 9 =

### Controlling output diversity with temperature scaling
* starting with temperature scaling

* ok, now  ---> process of selecting the token

In [62]:
### let's see, how we can get the next token...
prompt_example = "large language models are"
# first will convert the input prompt -> to ids
input_ids = torch.tensor(tokenizer.encode(prompt_example), device=device).unsqueeze(0)

print(input_ids)

tensor([[16767,  4128,  4119,   525]], device='cuda:0')


In [63]:
with torch.inference_mode():
    next_token = model(input_ids,)[:, -1]

print(next_token.shape)  ##  [batch, vocab_size]



torch.Size([1, 151936])


In [64]:
next_token = torch.argmax(next_token)  ## with highest logit score

print(f"This is the token: {tokenizer.decode([next_token])}")
print(f"This is the token id: {next_token}")

This is the token:  trained
This is the token id: 16176


* now let's visualize this, we are not taking entire vocab, but only 100 tokens---> word "trained" is between

In [ ]:
import matplotlib.pyplot as plt

def plot_scores_bar(next_token_logits, start=16126, end=16226, arrow=True, ylabel="Logit Value"):

    ## vocab subsection(instead of all vocab---> use only 100 tokens, to visualize)
    x = torch.arange(start, end)

    # Select logits for the first sequence in the batch and the token IDs
    # in the range [start:end], then convert to float32 and move to CPU. [we have given token in this range of vocab]
    logits_section = next_token_logits[0, start:end].float().cpu()    

    ## plot logits